# Notebook 02 — Huấn Luyện Nhánh SA-LSTM & Tối Ưu Simulated Annealing (Tái Hiện Phương Pháp)

**Đồ án môn học:** IS403 — Phân Tích Dữ Liệu Kinh Doanh (Module 1: Dự báo giá đồng 1 phiên trước)  
**Tác giả:** Viên Đăng Khoa  
**Mục tiêu notebook:** Tái hiện nhánh **SA-LSTM** với Holdout trên 6 bộ đặc trưng và Walk-Forward 542 phiên chỉ trên `FS06_WTI_CU`. Mỗi fold/seed lưu model, scaler và metadata để tái lập.

### Mục lục
1. Cấu hình thực nghiệm đồng nhất & nạp dữ liệu chuẩn
2. Chia tập chống rò rỉ thời gian (`make_context`) & trọng số suy giảm (`sample_weights`)
3. Định nghĩa từng kiến trúc mô hình (tách cell: Baseline LSTM, Baseline SimpleRNN, Paper SA-LSTM, Reoptimized SA-LSTM)
4. Hàm huấn luyện, lưu trọng số/scaler & đánh giá thang giá gốc
5. Thuật toán tìm kiếm siêu tham số **Fresh Simulated Annealing (SA)**
6. Huấn luyện đầy đủ lưới Holdout (6 bộ FS × 4 tỷ lệ × 3 chiến lược × 4 mô hình = 288 fits) & lưu trọng số
7. Walk-Forward FS06 (9 folds × 5 seeds × 3 mô hình = 135 fits) và lưu artifact từng lượt
8. Đóng gói `02_sa_artifacts.zip` (nạp trực tiếp vào Notebook 04)

> **Lưu ý:** Toàn bộ kết quả thực nghiệm chính thức của đồ án đã được chạy hoàn tất và lưu trong báo cáo; notebook này hệ thống hóa lại toàn bộ mã nguồn sạch (self-contained) để nộp đồ án và bảo đảm khả năng tái lập trên Kaggle (`USE_CACHED_SA_HP = False`).

## 1. Cấu hình thực nghiệm & nạp dữ liệu gốc

In [ ]:
import gc
import glob
import hashlib
import json
import shutil
import time
import warnings
import zipfile
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")

# Thư mục xuất artifact trên Kaggle
OUT = Path("/kaggle/working/02_sa_artifacts") if Path("/kaggle/working").exists() else Path("02_sa_artifacts")
WEIGHTS_DIR = OUT / "holdout_weights"
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
for stale_name in ("sa_wf_dm_hac_by_seed.csv",):
    (OUT / stale_name).unlink(missing_ok=True)

# === THAM SỐ ĐỒNG KHỚP 100% VỚI NOTEBOOK 03 (NON-SA) ===
TARGET = "Copper"
LOOKBACK = 22
HORIZON = 1
BATCH_SIZE = 64
SEED = 42
VAL_FRACTION = 0.10
RATIOS = (0.80, 0.90, 0.925, 0.95)
SCHEMES = ("expanding", "rolling_756", "recency_weighted")
ROLLING_WINDOW = 756
RECENCY_HALF_LIFE = 252

# Cấu hình huấn luyện SA-series (trần 100 epochs theo Chen et al., 2023 + Early Stopping)
EPOCHS_HOLDOUT = 100
PATIENCE_HOLDOUT = 9
WF_EPOCHS = 45
WF_PATIENCE = 7
WF_STEP = 63
WF_SEEDS = (41, 42, 43, 44, 45)
RUN_WALK_FORWARD = True

# Cấu hình Simulated Annealing:
# Đặt False để trình bày đầy đủ quy trình chạy Fresh SA từ đầu theo đúng chuẩn tái lập.
# (Ghi chú: Kết quả thực nghiệm chính thức đã chạy xong trước đó; bộ HP tham chiếu lưu bên dưới.)
USE_CACHED_SA_HP = False
SA_EVALS = 16
SA_SEARCH_EPOCHS = 15
SA_PATIENCE = 4
SA_SEED = 2026
SA_SPACE = {"units_1": (16, 96), "units_2": (24, 160), "learning_rate": (1e-5, 3e-3)}
PAPER_SA_LSTM_HP = {"units_1": 39, "units_2": 111, "learning_rate": 5e-5}

# Bộ HP đã tìm ra từ các đợt thực nghiệm chính thức (chỉ dùng nếu đổi USE_CACHED_SA_HP = True)
REFERENCE_SA_HP = {
    "FS02_PAPER_CU":  {"units_1": 22, "units_2": 91,  "learning_rate": 0.000922},
    "FS03_PAPER_EXO": {"units_1": 22, "units_2": 91,  "learning_rate": 0.000922},
    "FS04_METALS_CU": {"units_1": 22, "units_2": 83,  "learning_rate": 0.000655},
    "FS05_METALS_EXO":{"units_1": 22, "units_2": 83,  "learning_rate": 0.000655},
    "FS06_WTI_CU":    {"units_1": 47, "units_2": 127, "learning_rate": 5.0478806754686805e-05},
}

# Đầy đủ 6 bộ đặc trưng, đồng bộ với Notebook 03 và Notebook 04
FEATURE_SETS = {
    "FS01_CU":         ["Copper"],
    "FS02_PAPER_CU":   ["Copper", "WTI", "Gold", "Silver"],
    "FS03_PAPER_EXO":  ["WTI", "Gold", "Silver"],
    "FS04_METALS_CU":  ["Copper", "Aluminium", "Tin", "Silver", "Gold"],
    "FS05_METALS_EXO": ["Aluminium", "Tin", "Silver", "Gold"],
    "FS06_WTI_CU":     ["Copper", "Aluminium", "Tin", "WTI"],
}
WF_FEATURE_SETS = ("FS06_WTI_CU",)

EXPECTED_SHA = "9580ede12d474c8f990556842d551dac176b3c16b2d6a4bffb7dc41dd54bf600"
candidates = (
    glob.glob("/kaggle/input/**/dataset_9_commodities_daily.csv", recursive=True)
    + glob.glob("**/dataset_9_commodities_daily.csv", recursive=True)
)
if not candidates:
    raise FileNotFoundError("Thiếu dataset_9_commodities_daily.csv. Hãy Add Data vào Kaggle.")

DATA_PATH = Path(candidates[0])
DATA_SHA = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
assert DATA_SHA == EXPECTED_SHA, "SHA-256 dữ liệu không khớp bộ chuẩn."

data = pd.read_csv(DATA_PATH)
data["Date"] = pd.to_datetime(data["Date"], errors="raise")
data = data.sort_values("Date").reset_index(drop=True)
assert len(data) == 2710 and data["Date"].is_unique

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
print(f"Sẵn sàng | GPU: {tf.config.list_physical_devices('GPU')} | Feature sets: {list(FEATURE_SETS)}")
print(f"Lưới Holdout: {len(FEATURE_SETS)} FS × {len(RATIOS)} splits × {len(SCHEMES)} schemes × 4 models = {len(FEATURE_SETS)*len(RATIOS)*len(SCHEMES)*4} fits")

## 2. Chia tập chống rò rỉ thời gian (`make_context`) & trọng số suy giảm (`sample_weights`)

In [ ]:
def reset_keras(seed: int = SEED) -> None:
    tf.keras.backend.clear_session()
    gc.collect()
    np.random.seed(int(seed))
    tf.keras.utils.set_random_seed(int(seed))


def sample_weights(length: int, scheme: str, half_life: int = RECENCY_HALF_LIFE):
    """Tạo trọng số hàm mũ ưu tiên phiên gần đây (half-life = 252 phiên ≈ 1 năm), chuẩn hóa trung bình = 1."""
    if scheme != "recency_weighted":
        return None
    ages = np.arange(length - 1, -1, -1, dtype=np.float32)
    w = np.exp(-np.log(2.0) * ages / float(half_life))
    return (w / np.mean(w)).astype("float32")


def make_context(frame: pd.DataFrame, ratio: float, scheme: str, feature_set: str, val_fraction: float = VAL_FRACTION) -> dict:
    """Tạo ngữ cảnh train/val/test theo đúng thứ tự thời gian; MinMaxScaler chỉ fit trên [fit_start, val_start)."""
    n = len(frame)
    split = int(n * ratio)
    val_start = int(split * (1.0 - val_fraction))
    fit_start = 0 if scheme != "rolling_756" else max(0, val_start - ROLLING_WINDOW)
    cols = FEATURE_SETS[feature_set]
    assert LOOKBACK + 25 < val_start < split < n
    assert (val_start - fit_start) > LOOKBACK + 10

    xsc = MinMaxScaler().fit(frame.iloc[fit_start:val_start][cols])
    ysc = MinMaxScaler().fit(frame.iloc[fit_start:val_start][[TARGET]])

    x_all = xsc.transform(frame[cols]).astype("float32")
    y_all = ysc.transform(frame[[TARGET]]).ravel().astype("float32")

    def build_block(start: int, end: int, restrict_train: bool = False) -> dict:
        indices = np.arange(max(start, LOOKBACK), end, dtype=int)
        if restrict_train:
            indices = indices[indices - LOOKBACK >= fit_start]
        X = np.stack([x_all[t - LOOKBACK : t] for t in indices]).astype("float32")
        return {
            "idx": indices,
            "date": frame["Date"].iloc[indices].to_numpy(),
            "X": X,
            "y": y_all[indices],
            "prev": y_all[indices - 1],
            "y_raw": frame[TARGET].iloc[indices].to_numpy(dtype=np.float64),
        }

    tr = build_block(fit_start, val_start, restrict_train=True)
    va = build_block(val_start, split)
    te = build_block(split, n)
    assert tr["X"].shape[1:] == (LOOKBACK, len(cols))

    return {
        "train": tr, "val": va, "test": te,
        "xsc": xsc, "ysc": ysc,
        "features": cols, "feature_set": feature_set,
        "ratio": float(ratio), "scheme": scheme,
        "fit_start": fit_start, "val_start": val_start, "test_start": split,
    }


def make_wf_fold_context(frame: pd.DataFrame, origin: int, end: int, feature_set: str) -> dict:
    """Tạo ngữ cảnh cho 1 fold Walk-Forward Expanding: chỉ dùng lịch sử < origin, tách 10% cuối làm validation."""
    val_start = int(origin * (1.0 - VAL_FRACTION))
    cols = FEATURE_SETS[feature_set]
    assert LOOKBACK + 30 < val_start < origin < end <= len(frame)

    xsc = MinMaxScaler().fit(frame.iloc[:val_start][cols])
    ysc = MinMaxScaler().fit(frame.iloc[:val_start][[TARGET]])
    x_all = xsc.transform(frame[cols]).astype("float32")
    y_all = ysc.transform(frame[[TARGET]]).ravel().astype("float32")

    def build_block(first: int, last: int) -> dict:
        ids = np.arange(max(first, LOOKBACK), last, dtype=int)
        return {
            "idx": ids,
            "date": frame["Date"].iloc[ids].to_numpy(),
            "X": np.stack([x_all[t - LOOKBACK : t] for t in ids]).astype("float32"),
            "y": y_all[ids],
            "prev": y_all[ids - 1],
            "y_raw": frame[TARGET].iloc[ids].to_numpy(dtype=np.float64),
        }

    return {
        "train": build_block(LOOKBACK, val_start),
        "val": build_block(val_start, origin),
        "test": build_block(origin, end),
        "xsc": xsc, "ysc": ysc,
        "features": cols, "feature_set": feature_set,
        "ratio": float(origin / len(frame)), "scheme": "expanding",
        "fit_start": 0, "val_start": val_start, "test_start": origin,
    }


# Kiểm tra nhanh tính toàn vẹn shape
_ctx_test = make_context(data, 0.80, "expanding", "FS06_WTI_CU")
print("PASS make_context (80/20 FS06):", _ctx_test["train"]["X"].shape, _ctx_test["val"]["X"].shape, _ctx_test["test"]["X"].shape)

## 3. Định nghĩa kiến trúc các mô hình nhánh SA-series (tách từng cell rõ ràng)

### 3.1 Mô hình đối chứng `Baseline_LSTM` — Input `(B, 22, F)` → Output `(B, 1)` (scaled)

In [ ]:
def build_baseline_lstm(input_shape: tuple, hp: dict = None) -> tf.keras.Model:
    """LSTM chuẩn 2 tầng (64 → 32), Adam LR=1e-3, hàm mất mát MSE trên thang đã scale."""
    model = tf.keras.Sequential(name="Baseline_LSTM")
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.LSTM(64, return_sequences=True))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.LSTM(32))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.Dense(16, activation="relu"))
    model.add(tf.keras.layers.Dense(1, activation="linear"))
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")
    return model

### 3.2 Mô hình đối chứng `Baseline_SimpleRNN` — Input `(B, 22, F)` → Output `(B, 1)` (scaled)

In [ ]:
def build_baseline_rnn(input_shape: tuple, hp: dict = None) -> tf.keras.Model:
    """SimpleRNN 2 tầng (64 → 32) không cổng nhớ, cùng đầu ra Dense(16 → 1)."""
    model = tf.keras.Sequential(name="Baseline_SimpleRNN")
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.SimpleRNN(64, return_sequences=True))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.SimpleRNN(32))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.Dense(16, activation="relu"))
    model.add(tf.keras.layers.Dense(1, activation="linear"))
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse")
    return model

### 3.3 Mô hình tái hiện `Paper_SA_LSTM` (Chen et al., 2023) — `LSTM(39 → 111)`, `LR = 5e-5`

In [ ]:
def build_paper_sa_lstm(input_shape: tuple, hp: dict = None) -> tf.keras.Model:
    """Tái dựng kiến trúc SA-LSTM bài báo Chen et al. (2023): LSTM(39→111), LR=5e-5.
    Ghi chú tái dựng: tầng Dense(32, relu) là giả định triển khai được công bố minh bạch."""
    model = tf.keras.Sequential(name="Paper_SA_LSTM")
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.LSTM(39, return_sequences=True, activation="tanh", recurrent_activation="sigmoid"))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.LSTM(111, activation="tanh", recurrent_activation="sigmoid"))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.Dense(32, activation="relu"))
    model.add(tf.keras.layers.Dense(1, activation="linear"))
    model.compile(optimizer=tf.keras.optimizers.Adam(5e-5), loss="mse")
    return model

### 3.4 Mô hình tối ưu `Reoptimized_SA_LSTM` — nhận bộ `(units_1, units_2, learning_rate)` từ Fresh SA

In [ ]:
def build_reopt_sa_lstm(input_shape: tuple, hp: dict = None) -> tf.keras.Model:
    """SA-LSTM nhận bộ siêu tham số (units_1, units_2, learning_rate) tối ưu bởi Simulated Annealing."""
    cfg = hp or PAPER_SA_LSTM_HP
    u1 = int(cfg["units_1"])
    u2 = int(cfg["units_2"])
    lr = float(cfg["learning_rate"])

    model = tf.keras.Sequential(name="Reoptimized_SA_LSTM")
    model.add(tf.keras.layers.Input(shape=input_shape))
    model.add(tf.keras.layers.LSTM(u1, return_sequences=True, activation="tanh", recurrent_activation="sigmoid"))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.LSTM(u2, activation="tanh", recurrent_activation="sigmoid"))
    model.add(tf.keras.layers.Dropout(0.20))
    model.add(tf.keras.layers.Dense(32, activation="relu"))
    model.add(tf.keras.layers.Dense(1, activation="linear"))
    model.compile(optimizer=tf.keras.optimizers.Adam(lr), loss="mse")
    return model

## 4. Hàm huấn luyện, lưu trọng số/scaler & tính thước đo thang giá gốc

In [ ]:
def fit_predict_and_save(
    builder,
    ctx: dict,
    model_name: str,
    hp: dict = None,
    seed: int = SEED,
    epochs: int = EPOCHS_HOLDOUT,
    patience: int = PATIENCE_HOLDOUT,
    skip_test: bool = False,
    save_dir: Path = None,
    meta_extras: dict = None,
) -> dict:
    """Huấn luyện 1 mô hình nhánh Direct (thang scaled), tùy chọn lưu trọng số .keras + scaler .joblib."""
    reset_keras(seed)
    tr, va, te = ctx["train"], ctx["val"], ctx["test"]
    model = builder(tr["X"].shape[1:], hp)
    weights = sample_weights(len(tr["idx"]), ctx["scheme"])

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=patience, restore_best_weights=True, verbose=0
        )
    ]

    t0 = time.monotonic()
    hist = model.fit(
        tr["X"], tr["y"],
        validation_data=(va["X"], va["y"]),
        sample_weight=weights,
        epochs=epochs,
        batch_size=BATCH_SIZE,
        shuffle=False,
        verbose=0,
        callbacks=callbacks,
    )
    fit_sec = float(time.monotonic() - t0)

    val_pred = model.predict(va["X"], verbose=0).ravel()
    test_pred = None if skip_test else model.predict(te["X"], verbose=0).ravel()
    val_mse = float(mean_squared_error(va["y"], val_pred))

    # Lưu trọng số + scaler + metadata cho Notebook 04 tái đánh giá độc lập
    if save_dir is not None:
        save_dir.mkdir(parents=True, exist_ok=True)
        model.save(save_dir / "model.keras")
        joblib.dump(ctx["xsc"], save_dir / "xsc.joblib")
        joblib.dump(ctx["ysc"], save_dir / "ysc.joblib")
        meta = {
            "model": model_name,
            "target_mode": "direct",
            "feature_set": ctx["feature_set"],
            "features": ctx["features"],
            "ratio": ctx["ratio"],
            "scheme": ctx["scheme"],
            "seed": int(seed),
            "hp": hp or {},
            "epochs_run": len(hist.history["loss"]),
        }
        if meta_extras:
            meta.update(meta_extras)
        (save_dir / "meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")

    del model
    reset_keras(seed)
    return {
        "val_pred": val_pred,
        "test_pred": test_pred,
        "val_mse": val_mse,
        "epochs_run": len(hist.history["loss"]),
        "fit_seconds": fit_sec,
    }


def evaluate_predictions(ctx: dict, model_name: str, val_pred: np.ndarray, test_pred: np.ndarray, seed: int = SEED, extras: dict = None):
    """Nghịch đảo dự báo về đơn vị giá đồng gốc (USD/lb) và tính MAE, RMSE, MAPE, R2."""
    ysc = ctx["ysc"]
    te, va = ctx["test"], ctx["val"]

    pred_raw = ysc.inverse_transform(np.asarray(test_pred).reshape(-1, 1)).ravel()
    y_true = te["y_raw"]

    mae = float(mean_absolute_error(y_true, pred_raw))
    rmse = float(np.sqrt(mean_squared_error(y_true, pred_raw)))
    mape = float(np.mean(np.abs((y_true - pred_raw) / y_true)) * 100.0)
    r2 = float(r2_score(y_true, pred_raw))

    row = {
        "protocol": "SA_series",
        "model": model_name,
        "feature_set": ctx["feature_set"],
        "scheme": ctx["scheme"],
        "ratio": ctx["ratio"],
        "seed": int(seed),
        "n_test": len(y_true),
        "mae": mae,
        "rmse": rmse,
        "mape_pct": mape,
        "r2": r2,
        "val_mse_scaled": float(mean_squared_error(va["y"], val_pred)),
    }
    if extras:
        row.update(extras)

    pred_df = pd.DataFrame({
        "Date": te["date"],
        "ratio": ctx["ratio"],
        "scheme": ctx["scheme"],
        "feature_set": ctx["feature_set"],
        "model": model_name,
        "seed": int(seed),
        "actual": y_true,
        "pred": pred_raw,
    })
    return row, pred_df

## 5. Thuật toán tìm kiếm siêu tham số Fresh Simulated Annealing (SA)
- Tìm bộ `(units_1, units_2, learning_rate)` riêng cho từng bộ đặc trưng trên tập validation của mốc tham chiếu `80/20 Expanding` (`skip_test=True`, tuyệt đối không nhìn tập test).
- Lịch làm nguội: `T_k = max(0.05, 0.88^k)`, ngân sách 16 trials × 15 epochs/trial.

In [ ]:
def clip_hp(h: dict) -> dict:
    return {
        "units_1": int(np.clip(round(h["units_1"]), *SA_SPACE["units_1"])),
        "units_2": int(np.clip(round(h["units_2"]), *SA_SPACE["units_2"])),
        "learning_rate": float(np.clip(h["learning_rate"], *SA_SPACE["learning_rate"])),
    }


def sa_neighbor(h: dict, rng: np.random.Generator) -> dict:
    cand = dict(h)
    dim = rng.choice(["units_1", "units_2", "learning_rate"])
    if dim == "learning_rate":
        cand[dim] = float(cand[dim] * np.exp(rng.normal(0.0, 0.65)))
    else:
        cand[dim] = int(cand[dim] + rng.choice([-1, 1]) * rng.choice([8, 16, 24]))
    return clip_hp(cand)


def run_fresh_sa(feature_set: str, budget: int = SA_EVALS, seed: int = SA_SEED) -> dict:
    ctx = make_context(data, min(RATIOS), "expanding", feature_set)
    rng = np.random.default_rng(seed)
    current = clip_hp(PAPER_SA_LSTM_HP)
    best = dict(current)
    current_loss = float("inf")
    best_loss = float("inf")
    sa_log_path = OUT / "sa_search_trials.csv"

    for k in range(budget):
        temp = float(max(0.05, 1.0 * (0.88 ** k)))
        proposal = dict(current) if k == 0 else sa_neighbor(current, rng)
        res = fit_predict_and_save(
            build_reopt_sa_lstm, ctx, "reopt_sa_lstm_search",
            hp=proposal, seed=seed + k, epochs=SA_SEARCH_EPOCHS,
            patience=SA_PATIENCE, skip_test=True, save_dir=None,
        )
        score = float(res["val_mse"])
        eff_temp = max(1e-8, temp * max(score, current_loss if np.isfinite(current_loss) else score, 1e-5))
        accept = (score <= current_loss) or (rng.random() < np.exp(-max(0.0, score - current_loss) / eff_temp))
        if accept:
            current, current_loss = dict(proposal), score
        if score < best_loss:
            best, best_loss = dict(proposal), score

        entry = {
            "feature_set": feature_set, "trial": k + 1, "temperature": temp,
            "val_mse_scaled": score, "accepted": bool(accept), "best_val_mse_scaled": best_loss,
            "epochs_run": res["epochs_run"], "fit_seconds": res["fit_seconds"], **proposal,
        }
        pd.DataFrame([entry]).to_csv(sa_log_path, mode="a", header=not sa_log_path.exists(), index=False)
        print(f"SA {feature_set} [{k+1:02}/{budget}]: val={score:.6f} | best={best_loss:.6f} | acc={accept}", flush=True)
    return best


sa_best_hp = {}
if USE_CACHED_SA_HP:
    sa_best_hp = dict(REFERENCE_SA_HP)
    for idx_fs, fs_name in enumerate(sorted(set(FEATURE_SETS) - set(sa_best_hp))):
        sa_best_hp[fs_name] = run_fresh_sa(fs_name, budget=SA_EVALS, seed=SA_SEED + idx_fs * 1000)
    print("Đang dùng HP tham chiếu; feature set chưa có cache được chạy SA mới.")
else:
    if (OUT / "sa_search_trials.csv").exists():
        (OUT / "sa_search_trials.csv").unlink()
    for idx_fs, fs_name in enumerate(FEATURE_SETS):
        sa_best_hp[fs_name] = run_fresh_sa(fs_name, budget=SA_EVALS, seed=SA_SEED + idx_fs * 1000)

(OUT / "sa_best_hp.json").write_text(json.dumps(sa_best_hp, indent=2), encoding="utf-8")
display(pd.DataFrame([{"feature_set": k, **v} for k, v in sa_best_hp.items()]))

## 6. Huấn luyện đầy đủ lưới Holdout (6 bộ FS × 4 tỷ lệ × 3 chiến lược × 4 mô hình = 288 fits) & lưu trọng số

In [ ]:
SA_MODELS = [
    ("baseline_lstm", build_baseline_lstm),
    ("baseline_rnn",  build_baseline_rnn),
    ("paper_sa_lstm", build_paper_sa_lstm),
    ("reopt_sa_lstm", build_reopt_sa_lstm),
]

HOLD_RES_CSV = OUT / "sa_holdout_results.csv"
HOLD_PRED_CSV = OUT / "sa_holdout_predictions.csv"
for _p in (HOLD_RES_CSV, HOLD_PRED_CSV):
    if _p.exists():
        _d = pd.read_csv(_p).drop(columns=["naive", "mae_naive"], errors="ignore")
        _d.to_csv(_p, index=False)

done_keys = set()
if HOLD_RES_CSV.exists():
    prev_df = pd.read_csv(HOLD_RES_CSV)
    done_keys = set(zip(prev_df["ratio"].round(4), prev_df["feature_set"], prev_df["scheme"], prev_df["model"]))

jobs = [
    (ratio, fs, scheme, m_name, builder)
    for ratio in RATIOS
    for fs in FEATURE_SETS
    for scheme in SCHEMES
    for m_name, builder in SA_MODELS
]

print(f"Bắt đầu lưới Holdout SA-series: {len(done_keys)}/{len(jobs)} đã hoàn thành.")
for i, (ratio, fs, scheme, m_name, builder) in enumerate(jobs, 1):
    key = (round(ratio, 4), fs, scheme, m_name)
    if key in done_keys:
        continue

    ctx = make_context(data, ratio, scheme, fs)
    hp = PAPER_SA_LSTM_HP if m_name == "paper_sa_lstm" else (sa_best_hp[fs] if m_name == "reopt_sa_lstm" else None)
    tag = f"r{int(round(ratio*1000))}_{fs}_{scheme}_{m_name}_s{SEED}"
    save_dir = WEIGHTS_DIR / tag

    fit_out = fit_predict_and_save(
        builder, ctx, m_name, hp=hp, seed=SEED,
        epochs=EPOCHS_HOLDOUT, patience=PATIENCE_HOLDOUT,
        skip_test=False, save_dir=save_dir,
    )
    row, pred_df = evaluate_predictions(
        ctx, m_name, fit_out["val_pred"], fit_out["test_pred"], seed=SEED,
        extras={
            "epochs_run": fit_out["epochs_run"],
            "fit_seconds": fit_out["fit_seconds"],
            "weight_dir": str(save_dir.relative_to(OUT)),
            "hp_config": json.dumps(hp or {}, sort_keys=True),
        },
    )
    pd.DataFrame([row]).to_csv(HOLD_RES_CSV, mode="a", header=not HOLD_RES_CSV.exists(), index=False)
    pred_df.to_csv(HOLD_PRED_CSV, mode="a", header=not HOLD_PRED_CSV.exists(), index=False)
    done_keys.add(key)
    print(f"[{i:03}/{len(jobs)}] {ratio:.3f} | {fs:<15} | {scheme:<16} | {m_name:<14} | MAE={row['mae']:.5f} | R2={row['r2']:.4f}", flush=True)

assert len(done_keys) == len(jobs), f"Holdout SA chưa đủ: {len(done_keys)}/{len(jobs)}"
print("HOÀN TẤT LƯỚI HOLDOUT SA-SERIES & LƯU TRỌNG SỐ.")

## 7. Walk-Forward chỉ trên FS06_WTI_CU (542 phiên, 135 fits)
Mỗi lượt lưu tại `walkforward_weights/<FS>_<fold>_<model>_<seed>/` gồm `model.keras`, `xsc.joblib`, `ysc.joblib`, `meta.json`. Toàn bộ được đóng gói trong `02_sa_artifacts.zip`.
- Đồng khớp 100% lịch ngày (`07/10/2024 → 13/11/2026`), bước huấn luyện lại `RETRAIN_EVERY = 63` phiên, `WF_EPOCHS = 45`, `WF_PATIENCE = 7` với Notebook 03.
- Chạy 3 mô hình riêng của nhánh SA-series: `baseline_rnn`, `paper_sa_lstm`, `reopt_sa_lstm` (`baseline_lstm` trùng kiến trúc với `lstm_direct` ở Notebook 03).

In [ ]:
WF_SA_MODELS = [
    ("baseline_rnn",  build_baseline_rnn),
    ("paper_sa_lstm", build_paper_sa_lstm),
    ("reopt_sa_lstm", build_reopt_sa_lstm),
]

WF_FOLD_CSV = OUT / "sa_wf_fold_metrics.csv"
WF_PRED_CSV = OUT / "sa_wf_predictions.csv"
WF_WEIGHTS_DIR = OUT / "walkforward_weights"
WF_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
for _d in WF_WEIGHTS_DIR.iterdir():
    if _d.is_dir() and not _d.name.startswith("FS06_WTI_CU_"):
        shutil.rmtree(_d)
for _p in (WF_FOLD_CSV, WF_PRED_CSV):
    if _p.exists():
        _d = pd.read_csv(_p)
        _d = _d[_d["feature_set"].isin(WF_FEATURE_SETS)].drop(columns=["naive", "mae_naive"], errors="ignore")
        _d.to_csv(_p, index=False)

if RUN_WALK_FORWARD:
    first_idx = int(len(data) * 0.80)  # 2168 -> 542 phiên test
    folds = [(s, min(s + WF_STEP, len(data))) for s in range(first_idx, len(data), WF_STEP)]

    wf_done = set()
    if WF_FOLD_CSV.exists():
        old_wf = pd.read_csv(WF_FOLD_CSV)
        wf_done = set(zip(old_wf["feature_set"], old_wf["fold_start"].astype(int), old_wf["seed"].astype(int), old_wf["model"]))

    total_wf = len(WF_FEATURE_SETS) * len(folds) * len(WF_SEEDS) * len(WF_SA_MODELS)
    print(f"Walk-Forward SA-series: {len(WF_FEATURE_SETS)} FS × {len(folds)} folds × {len(WF_SEEDS)} seeds × {len(WF_SA_MODELS)} models = {total_wf} fits")

    for fs in WF_FEATURE_SETS:
        for fi, (f_start, f_end) in enumerate(folds, 1):
            ctx = make_wf_fold_context(data, f_start, f_end, fs)
            for seed in WF_SEEDS:
                for m_name, builder in WF_SA_MODELS:
                    tag = f"{fs}_f{fi:02d}_{f_start}_{f_end}_{m_name}_s{seed}"
                    save_dir = WF_WEIGHTS_DIR / tag
                    artifact_ok = all((save_dir / n).exists() for n in ("model.keras", "xsc.joblib", "ysc.joblib", "meta.json"))
                    if (fs, f_start, seed, m_name) in wf_done and artifact_ok:
                        continue
                    hp = PAPER_SA_LSTM_HP if m_name == "paper_sa_lstm" else (sa_best_hp[fs] if m_name == "reopt_sa_lstm" else None)
                    fit_out = fit_predict_and_save(
                        builder, ctx, m_name, hp=hp, seed=seed,
                        epochs=WF_EPOCHS, patience=WF_PATIENCE,
                        skip_test=False, save_dir=save_dir,
                        meta_extras={"protocol": "walk_forward", "fold_number": fi, "fold_start": f_start, "fold_end": f_end},
                    )
                    row, pred_df = evaluate_predictions(
                        ctx, m_name, fit_out["val_pred"], fit_out["test_pred"], seed=seed,
                        extras={"fold_number": fi, "fold_start": f_start, "fold_end": f_end, "epochs_run": fit_out["epochs_run"]},
                    )
                    pred_df.to_csv(WF_PRED_CSV, mode="a", header=not WF_PRED_CSV.exists(), index=False)
                    pd.DataFrame([row]).to_csv(WF_FOLD_CSV, mode="a", header=not WF_FOLD_CSV.exists(), index=False)
                    wf_done.add((fs, f_start, seed, m_name))
                    print(f"WF {fs} | Fold {fi}/{len(folds)} | seed={seed} | {m_name:<14} | MAE={row['mae']:.5f}", flush=True)

    # Tổng hợp theo feature set, model và seed
    _folds = pd.read_csv(WF_FOLD_CSV).drop_duplicates(["feature_set", "fold_start", "seed", "model"], keep="last")
    _folds.to_csv(WF_FOLD_CSV, index=False)
    wfp = pd.read_csv(WF_PRED_CSV).drop_duplicates(["Date", "feature_set", "model", "seed"])
    wf_counts = wfp.groupby(["feature_set", "model", "seed"]).size()
    assert len(wf_counts) == len(WF_FEATURE_SETS) * len(WF_SA_MODELS) * len(WF_SEEDS)
    assert wf_counts.eq(len(data) - first_idx).all(), "Mỗi nhóm Walk-Forward SA phải phủ đủ 542 ngày"
    wf_artifacts = list(WF_WEIGHTS_DIR.glob("*/meta.json"))
    assert len(wf_artifacts) == len(folds) * len(WF_SEEDS) * len(WF_SA_MODELS), f"Thiếu artifact WF SA: {len(wf_artifacts)}"
    seed_rows = []
    for (fs, m_name, seed), g in wfp.groupby(["feature_set", "model", "seed"]):
        ae = np.abs(g["actual"] - g["pred"])
        seed_rows.append({
            "feature_set": fs, "model": m_name, "seed": int(seed), "n": len(g),
            "mae": float(ae.mean()),
            "rmse": float(np.sqrt(np.mean((g["actual"] - g["pred"]) ** 2))),
            "mape_pct": float(np.mean(ae / np.abs(g["actual"])) * 100.0),
            "r2": float(r2_score(g["actual"], g["pred"])),
        })

    pd.DataFrame(seed_rows).to_csv(OUT / "sa_wf_seed_summary.csv", index=False)
    display(pd.DataFrame(seed_rows).groupby(["feature_set", "model"])[["mae", "rmse", "r2"]].agg(["mean", "std"]).round(5))

## 8. Xuất manifest & đóng gói `02_sa_artifacts.zip` (nạp vào Notebook 04)

In [ ]:
manifest = {
    "notebook": "02_Train_SA_LSTM_Series.ipynb",
    "dataset_sha256": DATA_SHA,
    "lookback": LOOKBACK,
    "horizon": HORIZON,
    "ratios": list(RATIOS),
    "schemes": list(SCHEMES),
    "feature_sets": FEATURE_SETS,
    "use_cached_sa_hp": USE_CACHED_SA_HP,
    "sa_best_hp": sa_best_hp,
    "epochs_holdout": EPOCHS_HOLDOUT,
    "wf_enabled": RUN_WALK_FORWARD,
    "wf_feature_sets": list(WF_FEATURE_SETS),
    "wf_seeds": list(WF_SEEDS),
    "wf_step": WF_STEP,
    "wf_weights_dir": "walkforward_weights",
    "wf_artifacts_expected": 135,
}
(OUT / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

zip_path = OUT.parent / "02_sa_artifacts.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for f in sorted(OUT.rglob("*")):
        if f.is_file():
            zf.write(f, arcname=f.relative_to(OUT))

with zipfile.ZipFile(zip_path, "r") as zf:
    archived_wf = [n for n in zf.namelist() if n.startswith("walkforward_weights/") and n.endswith("/meta.json")]
assert len(archived_wf) == 135, f"ZIP thiếu artifact WF SA: {len(archived_wf)}/135"
print(f"HOÀN TẤT ĐÓNG GÓI: {zip_path} ({zip_path.stat().st_size / (1024*1024):.2f} MB)")